# A Label Is Not Ground Truth

Article #32: nine synthetic experiments, nine original figures, eighteen canonical evidence CSVs. A latent outcome, an annotator vote, a policy-observed label, and a model prediction are *not* interchangeable variables.


## The label is a measurement channel

For class-conditional flip rates eta0=P(tildeY=1|Y=0) and eta1=P(tildeY=0|Y=1), the recorded positive fraction q obeys q=eta0+(1-eta0-eta1)*pi. An inversion requires **known or identified** rates and eta0+eta1 != 1. The synthetic studies use positively oriented channels with eta0+eta1<1. The generator supplies these rates; real annotation records usually do not.

Dawid–Skene EM assumes worker votes are independent *given* the latent class. Correlated workers can agree confidently on a shared mistake.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import numpy as np
from IPython.display import Image, display
# The kernel may start in the article folder or repository root.
# For an unrelated server directory, set LABEL_STUDY_DIR explicitly.
slug = 'a-label-is-not-ground-truth'
configured = os.environ.get('LABEL_STUDY_DIR')
location = Path.cwd().resolve()
candidates = ([Path(configured).expanduser().resolve()] if configured else
              [candidate for parent in (location, *location.parents)
               for candidate in (parent, parent/slug)])
study_dir = next((candidate for candidate in candidates
                  if (candidate/'analysis.py').is_file() and
                  (candidate/'a_label_is_not_ground_truth.ipynb').is_file()), None)
if study_dir is None:
    raise FileNotFoundError('Cannot find article analysis; set LABEL_STUDY_DIR')
sys.path.insert(0, str(study_dir))
from analysis import (Design, EXPERIMENTS, FIGURES, SEED, invert_noisy_prevalence, dawid_skene_binary, run)
print('Seed', SEED, '| studies', len(EXPERIMENTS))


In [ ]:
assert abs(invert_noisy_prevalence(.2125, .05, .30)-.25) < 1e-12
votes = np.tile([1,1,1,0,0], (40,1)).astype(int)
posterior = dawid_skene_binary(votes)
assert posterior.shape == (40,) and np.isfinite(posterior).all()
print('Class-conditional inversion and EM numerical checks passed')


## Execute independent quick studies without overwriting evidence

This notebook uses three independent outer repetitions per study. Canonical full mode uses 20–36. Quick outputs are created in a temporary directory with `_quick` suffixes; the committed full-mode CSV and PNG evidence is untouched. Plot bars represent Monte Carlo precision over independent outer repetitions, not observational confidence intervals.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory)
    summaries = run(Design(root=destination, quick=True))
    tables = sorted((destination/'results').glob('*_quick.csv'))
    figures = sorted((destination/'figures').glob('*_quick.png'))
    assert len(tables) == 18 and len(figures) == 9
    print('Tables:', len(tables), 'Figures:', len(figures))
    for study, figure in zip(EXPERIMENTS, FIGURES):
        print('\n', study, '| summary cells:', len(summaries[study]))
        display(Image(filename=str(destination/'figures'/f'{figure}_quick.png')))


## Interpret carefully

- Synthetic latent truth is available **only** because the generator created it. Label correction rates are not known just because a classifier assigns confident scores.
- The class-conditional correction is oracle, not a free identification method.
- Dawid–Skene here has complete votes, binary classes, pseudo-count regularization and an orientation anchored by initial majority vote. Shared worker errors break its model.
- Randomized audit sampling protects against selective labels; an audit restricted to flagged cases cannot identify global prevalence.
- A test label constructed in favor of model B is deliberate evaluation bias, not evidence that B generalized better.
- Row-level splits can reuse the same entity in train and test; the appropriate unit of independence depends on deployment.

Read `article.md` for derivations, sources, limitations, and a practical labeling-governance protocol.
